## Classes definitions

In [8]:
class Room:
    def __init__(self,name,capacity):
        self.name = name
        self.capacity = capacity

class Timeslot:
    def __init__(self,date: str):
        '''
        format of the date string must be "YYYY-MM-DD HH:MM"
        '''
        self.date = date

        data = date.split(' ')
        a = data[1].split(':')
        b = data[0].split('-')

        self.hour, self.minute = int(a[0]) , int(a[1])
        self.year, self.month, self.day = int(b[0]), int(b[1]), int(b[2])

        self.is_late = self.hour >= 17

    def __eq__(self, value):
        return self.date == value.date
    
    

## Genetic Algorithm Solution

In [9]:
import random
from time import sleep

class Chromosome:
    def __init__(self,size,domain,mutation_probability,dna = None):
        self.mutation = mutation_probability
        self.size = size
        self.domain = domain
        if dna is None:
            self.dna = [random.choice(domain) for _ in range(size)]
        else:
            self.dna = dna

    def mutate(self):
        for i in range(self.size):
            r = random.random()
            if r <= self.mutation:
                self.dna[i] = random.choice(self.domain)


class GeneticAlgorithm:
    def __init__(self,population: list[Chromosome],rooms_timeslots: list[tuple],exam_students: list[set]):
        self.population = population
        self.room_timeslot = rooms_timeslots
        self.exam_students = exam_students

    def crossover(self,chromo1 : Chromosome,chromo2 : Chromosome):
        crossover_point = chromo1.size // 2
        child1 = Chromosome(chromo1.size,chromo1.domain,chromo1.mutation,chromo1.dna[0:crossover_point] + chromo2.dna[crossover_point:])
        child2 = Chromosome(chromo1.size,chromo1.domain,chromo1.mutation,chromo2.dna[0:crossover_point] + chromo1.dna[crossover_point:])

        child1.mutate()
        child2.mutate()
        return child1 , child2

    def fitness(self,chromo: Chromosome):
        ## HARD CONSTRAINTS

        # check for two different exams having the same (room,timeslot)
        for i in range(chromo.size):
            for j in range(i+1,chromo.size):
                if chromo.dna[i] == chromo.dna[j]:
                    return 0
        # check for same students sitting for two different exams at the same timeslot
        for i in range(chromo.size):
           for j in range(i+1, chromo.size):
               same_students = len(self.exam_students[i] & self.exam_students[j]) > 0
               same_timeslot = self.room_timeslot[chromo.dna[i]][1] == self.room_timeslot[chromo.dna[j]][1]
               if same_students and same_timeslot:
                   return 0

        # check for room capacity violation (hard constraint)
        for i in range(chromo.size):
            if self.room_timeslot[chromo.dna[i]][0].capacity < len(self.exam_students[i]):
                return 0

        ## SOFT CONSTRAINTS

        # find the number of students who have consecutive exams (counts duplicates)
        # meaning if the same student has three different exams on the same day it will count that student twice
        # DEFINITION: a student has consecutive exams if he/she has two or more different exams on the same day
        consecutive_exams_penalty = 0
        total_mutual_students = 0
        for i in range(chromo.size):
            for j in range(i+1,chromo.size):
                overlap = len(self.exam_students[i] & self.exam_students[j])
                total_mutual_students += overlap
                if self.room_timeslot[chromo.dna[i]][1].day == self.room_timeslot[chromo.dna[j]][1].day:
                    consecutive_exams_penalty += overlap

        late_exams = 0
        total_exams = chromo.size
        for i in range(chromo.size):
            if self.room_timeslot[chromo.dna[i]][1].is_late:
                late_exams += 1

        # efficient allocation utility function
        # n : number of students
        # c : room capacity
        # f(n,c) = - (4n(c-n)^2)/c^2 outputs a number between 0 and 1 as a metric of how efficiently the room
        efficient_allocation_factor = 0
        for i in range(chromo.size):
            n = len(self.exam_students[i])
            c = self.room_timeslot[chromo.dna[i]][0].capacity
            efficient_allocation_factor += -4*n*(n-c)/(c**2)

        if total_mutual_students == 0:
            penalty_factor = 1 + late_exams / total_exams
        else:
            penalty_factor = 1 + late_exams / total_exams + 2*(consecutive_exams_penalty / total_mutual_students)

        fitness_value = (efficient_allocation_factor / total_exams) / penalty_factor
        return fitness_value

    def selection(self):
        population_fitness = [self.fitness(p) for p in self.population]

        # elitism: always keep best chromosome
        next_population = []
        best_index = max(range(len(self.population)), key=lambda i: population_fitness[i])
        next_population.append(self.population[best_index])

        # roulette wheel parent selection
        total_fitness = sum(population_fitness)
        target_parent_count = max(2, len(self.population) // 2)

        while len(next_population) < target_parent_count:
            if total_fitness == 0:
                selected = random.choice(self.population)
            else:
                r = random.uniform(0, total_fitness)
                acc = 0
                selected = self.population[-1]
                for chromo, fit in zip(self.population, population_fitness):
                    acc += fit
                    if acc >= r:
                        selected = chromo
                        break
            next_population.append(selected)

        if len(next_population) % 2 == 1:
            next_population.append(random.choice(next_population))

        children = []
        while len(next_population) + len(children) < len(self.population):
            parent1, parent2 = random.sample(next_population, 2)
            child1, child2 = self.crossover(parent1, parent2)
            children.extend([child1, child2])

        self.population = (next_population + children)[:len(self.population)]
        return self.population

    def run(self, generations: int,printer = False):
        for epoch in range(generations):
            if printer:
                print("epoch #",epoch," best solution fitness value: ", max([self.fitness(p) for p in self.population]), "\n")
                sleep(0.5)
            self.selection()
        return max(self.population, key=self.fitness)
    
# fitness function outputs 0 when efficient_allocation = 0 even though its a valid solution fix it

## CSP SOLUTION

In [10]:
class CSP:
    def __init__(self,exam_students,room_timeslot):
        self.exam = [None for _ in range(len(exam_students))]
        self.number_of_exams = len(exam_students)
        self.exam_students = exam_students
        self.room_timeslot = room_timeslot

        self.domain = {i: set(range(len(self.room_timeslot))) for i in range(self.number_of_exams)}

    def binary_hard_constraint(self,e1,e2,v1,v2):
        # no two exams can have the same (room,timeslot)
        if v1 == v2:
            return False
        # no student can sit for two different exams at the same time
        if len(self.exam_students[e1]&self.exam_students[e2]) > 0 and self.room_timeslot[v1][1] == self.room_timeslot[v2][1]:
            return False

        return True

    def unary_hard_constraint(self,e1,v1):
        # room capacity must exceed number of students
        if len(self.exam_students[e1]) > self.room_timeslot[v1][0].capacity:
            return False

        return True

    def run(self, time_limit_sec=5.0):
        import time

        def solution_fitness(schedule):
            # schedule has room_timeslot indices for every exam
            consecutive_exams_penalty = 0
            total_mutual_students = 0
            for i in range(self.number_of_exams):
                for j in range(i+1,self.number_of_exams):
                    overlap = len(self.exam_students[i] & self.exam_students[j])
                    total_mutual_students += overlap
                    if self.room_timeslot[schedule[i]][1].day == self.room_timeslot[schedule[j]][1].day:
                        consecutive_exams_penalty += overlap

            late_exams = 0
            total_exams = self.number_of_exams
            for i in range(self.number_of_exams):
                if self.room_timeslot[schedule[i]][1].is_late:
                    late_exams += 1

            efficient_allocation_factor = 0
            for i in range(self.number_of_exams):
                n = len(self.exam_students[i])
                c = self.room_timeslot[schedule[i]][0].capacity
                efficient_allocation_factor += -4*n*(n-c)/(c**2)

            if total_mutual_students == 0:
                penalty_factor = 1 + late_exams / total_exams
            else:
                penalty_factor = 1 + late_exams / total_exams + 2*(consecutive_exams_penalty / total_mutual_students)

            fitness_value = (efficient_allocation_factor / total_exams) / penalty_factor
            return fitness_value

        start = time.perf_counter()
        deadline = start + time_limit_sec
        nodes_explored = 0

        best_assignment = None
        best_fitness = float('-inf')

        def solve(assignment: list, not_assigned: set, domains: dict):
            nonlocal nodes_explored, best_assignment, best_fitness

            if time.perf_counter() >= deadline:
                return

            nodes_explored += 1

            if len(not_assigned) == 0:
                sol_fitness = solution_fitness(assignment)
                if sol_fitness > best_fitness:
                    best_fitness = sol_fitness
                    best_assignment = assignment.copy()
                return

            # Minimum Remaining Values (MRV)
            curr = min(not_assigned, key=lambda i: len(domains[i]))
            not_assigned.remove(curr)

            # Value ordering: prefer non-late slots and tighter capacity fit
            values = sorted(
                domains[curr],
                key=lambda v: (
                    self.room_timeslot[v][1].is_late,
                    abs(self.room_timeslot[v][0].capacity - len(self.exam_students[curr]))
                )
            )

            for possible_value in values:
                if time.perf_counter() >= deadline:
                    break

                if not self.unary_hard_constraint(curr,possible_value):
                    continue

                # deep copy of domains dict
                new_domain = {k: s.copy() for k, s in domains.items()}

                invalid_possible_value = False
                for e in not_assigned:
                    # collect removals first, then apply
                    to_remove = set()
                    for v in new_domain[e]:
                        if not self.binary_hard_constraint(curr,e,possible_value,v):
                            to_remove.add(v)

                    if to_remove:
                        new_domain[e] -= to_remove

                    if len(new_domain[e]) == 0: # empty domain so backtrack
                        invalid_possible_value = True
                        break

                if invalid_possible_value:
                    continue

                assignment[curr] = possible_value
                solve(assignment,not_assigned,new_domain)
                assignment[curr] = None

            not_assigned.add(curr)

        init_not_assigned = set(range(self.number_of_exams))
        init_assignment = [None for _ in range(self.number_of_exams)]
        solve(assignment=init_assignment, not_assigned=init_not_assigned, domains=self.domain)

        if best_assignment is not None:
            self.exam = best_assignment

        elapsed = time.perf_counter() - start
        print(f'CSP search explored {nodes_explored} nodes in {elapsed:.2f}s; best fitness so far: {best_fitness:.6f}' if best_assignment is not None else f'CSP search explored {nodes_explored} nodes in {elapsed:.2f}s; no feasible solution found yet')

        return self.exam if best_assignment is not None else None

## Demo Setup

In [11]:
# Build expanded demo data
rooms = [
    Room("R101", 30),
    Room("R102", 45),
    Room("R201", 60),
    Room("R202", 80),
    Room("Auditorium-A", 120),
    Room("Lab-B", 40),
]

timeslots = [
    Timeslot("2026-06-10 09:00"),
    Timeslot("2026-06-10 14:00"),
    Timeslot("2026-06-10 17:30"),
    Timeslot("2026-06-11 09:00"),
    Timeslot("2026-06-11 14:00"),
    Timeslot("2026-06-11 17:30"),
    Timeslot("2026-06-12 09:00"),
    Timeslot("2026-06-12 14:00"),
    Timeslot("2026-06-13 09:00"),
    Timeslot("2026-06-13 14:00"),
]

# Cartesian product: each gene maps to one (room, timeslot) pair
room_timeslot = [(room, slot) for room in rooms for slot in timeslots]

# Each set is one exam; values are student IDs taking that exam
exam_students = [
    {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12},
    {6, 7, 8, 9, 10, 13, 14, 15, 16, 17, 18, 19},
    {2, 4, 6, 8, 20, 21, 22, 23, 24, 25},
    {11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31},
    {1, 3, 5, 7, 9, 31, 32, 33, 34, 35},
    {15, 16, 17, 18, 19, 20, 35, 36, 37, 38, 39, 40},
    {22, 23, 24, 25, 26, 27, 41, 42, 43, 44, 45},
    {5, 10, 15, 20, 25, 30, 46, 47, 48, 49, 50},
    {3, 6, 9, 12, 15, 18, 21, 24, 51, 52, 53},
    {2, 5, 8, 11, 14, 17, 20, 54, 55, 56, 57},
    {1, 4, 7, 10, 13, 16, 19, 58, 59, 60, 61, 62},
    {27, 28, 29, 30, 31, 32, 33, 63, 64, 65, 66},
    {34, 35, 36, 37, 38, 39, 40, 67, 68, 69, 70},
    {41, 42, 43, 44, 45, 46, 47, 71, 72, 73, 74},
    {48, 49, 50, 51, 52, 53, 54, 75, 76, 77, 78},
    {55, 56, 57, 58, 59, 60, 61, 79, 80, 81, 82},
]

## Genetic Algorithm Test

In [12]:
# Create initial population
num_exams = len(exam_students)
domain = list(range(len(room_timeslot)))
population_size = 350
mutation_probability = 0.08

population = [
    Chromosome(
        size=num_exams,
        domain=domain,
        mutation_probability=mutation_probability,
    )
    for _ in range(population_size)
]

# Run GA
ga = GeneticAlgorithm(population, room_timeslot, exam_students)
best_ga = ga.run(generations=300, printer=False)

# print("Best fitness:", ga.fitness(best))
# print("\nBest schedule:")
# for exam_idx, gene in enumerate(best.dna, start=1):
#     room, slot = room_timeslot[gene]
#     print(
#         f"Exam {exam_idx}: {room.name} (cap={room.capacity}) at {slot.date} "
#         f"for {len(exam_students[exam_idx-1])} students"
#     )

## CSP Test

In [13]:
# Run CSP
csp = CSP(exam_students,room_timeslot)
best_csp = csp.run()

CSP search explored 84458 nodes in 5.00s; best fitness so far: 0.679334


## FINAL SOLUTION QUALITY REPORT

In [14]:
# Quality report for GA and CSP schedules
from collections import defaultdict

def generate_quality_report(solution, solution_name, fitness_value=None):
    """Generate a quality report for a schedule represented as room_timeslot indices."""
    if solution is None:
        print(f'--- {solution_name} Quality Report ---')
        print('No solution found!\n')
        return

    assigned = []
    unassigned_exam_indices = []
    for exam_idx, gene in enumerate(solution):
        if isinstance(gene, int) and 0 <= gene < len(room_timeslot):
            assigned.append((exam_idx, gene))
        else:
            unassigned_exam_indices.append(exam_idx)

    print(f'--- {solution_name} Quality Report ---')
    print('Assigned exams:', len(assigned), '/', len(solution))
    if unassigned_exam_indices:
        print('Unassigned/invalid exams:', len(unassigned_exam_indices))

    if not assigned:
        print('No valid assigned exams to evaluate.\n')
        return

    same_room_timeslot_conflicts = 0
    capacity_violations = 0
    student_same_timeslot_conflicts = 0

    used_pairs = set()
    for exam_idx, gene in assigned:
        if gene in used_pairs:
            same_room_timeslot_conflicts += 1
        used_pairs.add(gene)

        room, _ = room_timeslot[gene]
        if room.capacity < len(exam_students[exam_idx]):
            capacity_violations += 1

    for i in range(len(assigned)):
        exam_i, gene_i = assigned[i]
        _, slot_i = room_timeslot[gene_i]
        for j in range(i + 1, len(assigned)):
            exam_j, gene_j = assigned[j]
            _, slot_j = room_timeslot[gene_j]
            overlap = len(exam_students[exam_i] & exam_students[exam_j])
            if overlap > 0 and slot_i == slot_j:
                student_same_timeslot_conflicts += overlap

    late_exams = 0
    utilizations = []
    for exam_idx, gene in assigned:
        room, slot = room_timeslot[gene]
        if slot.is_late:
            late_exams += 1
        utilizations.append(len(exam_students[exam_idx]) / room.capacity)

    same_day_overlap_penalty = 0
    for i in range(len(assigned)):
        exam_i, gene_i = assigned[i]
        _, slot_i = room_timeslot[gene_i]
        for j in range(i + 1, len(assigned)):
            exam_j, gene_j = assigned[j]
            _, slot_j = room_timeslot[gene_j]
            if slot_i.day == slot_j.day:
                same_day_overlap_penalty += len(exam_students[exam_i] & exam_students[exam_j])

    slot_exam_counts = defaultdict(int)
    for _, gene in assigned:
        _, slot = room_timeslot[gene]
        slot_exam_counts[slot.date] += 1

    print('Hard constraints:')
    print('  Same room-timeslot conflicts:', same_room_timeslot_conflicts)
    print('  Capacity violations:', capacity_violations)
    print('  Student same-timeslot conflicts (overlap count):', student_same_timeslot_conflicts)

    print('\nSoft metrics:')
    print('  Late exams:', late_exams, '/', len(assigned))
    print('  Average room utilization:', round(sum(utilizations) / len(utilizations), 4))
    print('  Min room utilization:', round(min(utilizations), 4))
    print('  Max room utilization:', round(max(utilizations), 4))
    print('  Same-day overlap penalty:', same_day_overlap_penalty)

    print('\nTimeslot load:')
    for slot_date in sorted(slot_exam_counts):
        print(' ', slot_date, '->', slot_exam_counts[slot_date], 'exams')

    if fitness_value is not None:
        print('Fitness:', round(fitness_value, 6))

    print()

# Generate reports for both solutions
generate_quality_report(best_ga.dna, 'Genetic Algorithm', fitness_value=ga.fitness(best_ga))
generate_quality_report(best_csp, 'Constraint Satisfaction Problem')


--- Genetic Algorithm Quality Report ---
Assigned exams: 16 / 16
Hard constraints:
  Same room-timeslot conflicts: 0
  Capacity violations: 0
  Student same-timeslot conflicts (overlap count): 0

Soft metrics:
  Late exams: 0 / 16
  Average room utilization: 0.2882
  Min room utilization: 0.1833
  Max room utilization: 0.3667
  Same-day overlap penalty: 7

Timeslot load:
  2026-06-10 09:00 -> 2 exams
  2026-06-10 14:00 -> 2 exams
  2026-06-11 09:00 -> 2 exams
  2026-06-11 14:00 -> 2 exams
  2026-06-12 09:00 -> 3 exams
  2026-06-12 14:00 -> 1 exams
  2026-06-13 09:00 -> 2 exams
  2026-06-13 14:00 -> 2 exams
Fitness: 0.730643

--- Constraint Satisfaction Problem Quality Report ---
Assigned exams: 16 / 16
Hard constraints:
  Same room-timeslot conflicts: 0
  Capacity violations: 0
  Student same-timeslot conflicts (overlap count): 0

Soft metrics:
  Late exams: 0 / 16
  Average room utilization: 0.322
  Min room utilization: 0.2444
  Max room utilization: 0.4
  Same-day overlap penalty: 1

## Benchmark Run
Importing the benchmark and running the GA

In [17]:
from pathlib import Path
import re

def resolve_benchmark_dir():
    candidates = [
        Path("benchmark") / "1",
        Path("..") / "benchmark" / "1",
        Path.cwd() / "benchmark" / "1",
        Path.cwd() / ".." / "benchmark" / "1",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not locate benchmark/1 from the notebook working directory.")

benchmark_dir = resolve_benchmark_dir()

def load_benchmark():
    data_path = benchmark_dir / "data"

    rooms = []
    with data_path.open("r", encoding="utf-8") as f:
        lines = f.readlines()

    parsing_rooms = False
    for line in lines:
        stripped = line.strip()
        if not stripped:
            continue
        if stripped == "ROOMS":
            parsing_rooms = True
            continue
        if parsing_rooms and set(stripped) == {"-"}:
            continue
        if parsing_rooms and stripped.startswith("ROOM ASSIGNMENTS"):
            break
        if parsing_rooms:
            match = re.match(r"^([A-Z0-9-]+)\s+(\d+)", stripped)
            if match:
                rooms.append(Room(match.group(1), int(match.group(2))))

    dates = [
        "1995-01-23", "1995-01-24", "1995-01-25", "1995-01-26", "1995-01-27", "1995-01-28",
        "1995-01-30", "1995-01-31", "1995-02-01", "1995-02-02", "1995-02-03", "1995-02-04",
    ]
    times = ["09:00", "13:30", "16:30"]
    timeslots = []
    for date in dates:
        for time in times:
            if date in {"1995-01-28", "1995-02-04"} and time != "09:00":
                continue
            timeslots.append(Timeslot(f"{date} {time}"))

    exam_to_idx = {}
    with (benchmark_dir / "exams").open("r", encoding="utf-8") as f:
        for line in f:
            exam_code = line[:8].strip()
            if exam_code and exam_code not in exam_to_idx:
                exam_to_idx[exam_code] = len(exam_to_idx)

    exam_students = [set() for _ in range(len(exam_to_idx))]
    with (benchmark_dir / "enrolements").open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.split()
            if len(parts) >= 2:
                student, exam = parts[0], parts[1]
                exam_idx = exam_to_idx.get(exam)
                if exam_idx is not None:
                    exam_students[exam_idx].add(student)

    return rooms, timeslots, exam_students, exam_to_idx

def benchmark_fitness(solution, room_timeslot_data, exam_students_data):
    if solution is None:
        return float("-inf")

    consecutive_exams_penalty = 0
    total_mutual_students = 0
    for i in range(len(solution)):
        for j in range(i + 1, len(solution)):
            overlap = len(exam_students_data[i] & exam_students_data[j])
            total_mutual_students += overlap
            if room_timeslot_data[solution[i]][1].day == room_timeslot_data[solution[j]][1].day:
                consecutive_exams_penalty += overlap

    late_exams = sum(1 for gene in solution if room_timeslot_data[gene][1].is_late)
    total_exams = len(solution)

    efficient_allocation_factor = 0
    for i, gene in enumerate(solution):
        n = len(exam_students_data[i])
        c = room_timeslot_data[gene][0].capacity
        efficient_allocation_factor += -4 * n * (n - c) / (c ** 2)

    if total_mutual_students == 0:
        penalty_factor = 1 + late_exams / total_exams
    else:
        penalty_factor = 1 + late_exams / total_exams + 2 * (consecutive_exams_penalty / total_mutual_students)

    return (efficient_allocation_factor / total_exams) / penalty_factor

def generate_benchmark_quality_report(solution, solution_name, room_timeslot_data, exam_students_data, fitness_value=None):
    if solution is None:
        print(f'--- {solution_name} Quality Report ---')
        print('No solution found!\n')
        return

    assigned = []
    unassigned_exam_indices = []
    for exam_idx, gene in enumerate(solution):
        if isinstance(gene, int) and 0 <= gene < len(room_timeslot_data):
            assigned.append((exam_idx, gene))
        else:
            unassigned_exam_indices.append(exam_idx)

    print(f'--- {solution_name} Quality Report ---')
    print('Assigned exams:', len(assigned), '/', len(solution))
    if unassigned_exam_indices:
        print('Unassigned/invalid exams:', len(unassigned_exam_indices))

    if not assigned:
        print('No valid assigned exams to evaluate.\n')
        return

    same_room_timeslot_conflicts = 0
    capacity_violations = 0
    student_same_timeslot_conflicts = 0
    used_pairs = set()

    for exam_idx, gene in assigned:
        if gene in used_pairs:
            same_room_timeslot_conflicts += 1
        used_pairs.add(gene)

        room, _ = room_timeslot_data[gene]
        if room.capacity < len(exam_students_data[exam_idx]):
            capacity_violations += 1

    for i in range(len(assigned)):
        exam_i, gene_i = assigned[i]
        _, slot_i = room_timeslot_data[gene_i]
        for j in range(i + 1, len(assigned)):
            exam_j, gene_j = assigned[j]
            _, slot_j = room_timeslot_data[gene_j]
            overlap = len(exam_students_data[exam_i] & exam_students_data[exam_j])
            if overlap > 0 and slot_i == slot_j:
                student_same_timeslot_conflicts += overlap

    late_exams = 0
    utilizations = []
    for exam_idx, gene in assigned:
        room, slot = room_timeslot_data[gene]
        if slot.is_late:
            late_exams += 1
        utilizations.append(len(exam_students_data[exam_idx]) / room.capacity)

    same_day_overlap_penalty = 0
    for i in range(len(assigned)):
        exam_i, gene_i = assigned[i]
        _, slot_i = room_timeslot_data[gene_i]
        for j in range(i + 1, len(assigned)):
            exam_j, gene_j = assigned[j]
            _, slot_j = room_timeslot_data[gene_j]
            if slot_i.day == slot_j.day:
                same_day_overlap_penalty += len(exam_students_data[exam_i] & exam_students_data[exam_j])

    slot_exam_counts = {}
    for _, gene in assigned:
        _, slot = room_timeslot_data[gene]
        slot_exam_counts[slot.date] = slot_exam_counts.get(slot.date, 0) + 1

    print('Hard constraints:')
    print('  Same room-timeslot conflicts:', same_room_timeslot_conflicts)
    print('  Capacity violations:', capacity_violations)
    print('  Student same-timeslot conflicts (overlap count):', student_same_timeslot_conflicts)

    print('\nSoft metrics:')
    print('  Late exams:', late_exams, '/', len(assigned))
    print('  Average room utilization:', round(sum(utilizations) / len(utilizations), 4))
    print('  Min room utilization:', round(min(utilizations), 4))
    print('  Max room utilization:', round(max(utilizations), 4))
    print('  Same-day overlap penalty:', same_day_overlap_penalty)

    print('\nTimeslot load:')
    for slot_date in sorted(slot_exam_counts):
        print(' ', slot_date, '->', slot_exam_counts[slot_date], 'exams')

    if fitness_value is not None:
        print('Fitness:', round(fitness_value, 6))

    print()

# Load data
bm_rooms, bm_timeslots, bm_exam_students, bm_exam_to_idx = load_benchmark()
bm_room_timeslot = [(room, slot) for room in bm_rooms for slot in bm_timeslots]

print(f'Loaded Benchmark: {len(bm_rooms)} rooms, {len(bm_timeslots)} timeslots, {len(bm_room_timeslot)} room-timeslots options, and {len(bm_exam_students)} exams.')
print('Benchmark rooms sample:', ', '.join(f'{room.name}({room.capacity})' for room in bm_rooms[:5]))
print('Benchmark exams sample:', ', '.join(list(bm_exam_to_idx)[:5]))

Loaded Benchmark: 16 rooms, 32 timeslots, 512 room-timeslots options, and 800 exams.
Benchmark rooms sample: TRENT-HALL(125), TRENT-L19(80), TRENT-B46(40), COPSE-2(40), COPSE-3(50)
Benchmark exams sample: AA2016E1, AA3008E1, AAA011E1, AAA013E1, AAA022E1


In [18]:
# Setup GA for Benchmark
num_exams_bm = len(bm_exam_students)
domain_bm = list(range(len(bm_room_timeslot)))

# Using a smaller population and generations for the benchmark to run in a reasonable time
population_size_bm = 50
mutation_probability_bm = 0.05

population_bm = [
    Chromosome(
        size=num_exams_bm,
        domain=domain_bm,
        mutation_probability=mutation_probability_bm,
    )
    for _ in range(population_size_bm)
]

ga_bm = GeneticAlgorithm(population_bm, bm_room_timeslot, bm_exam_students)

print("Running GA on benchmark data...")
best_ga_bm = ga_bm.run(generations=10, printer=True)

print("\nGenerating report for Benchmark GA solution:")
generate_benchmark_quality_report(best_ga_bm.dna, 'Benchmark Genetic Algorithm', bm_room_timeslot, bm_exam_students, fitness_value=benchmark_fitness(best_ga_bm.dna, bm_room_timeslot, bm_exam_students))

Running GA on benchmark data...
epoch # 0  best solution fitness value:  0 

epoch # 1  best solution fitness value:  0 

epoch # 2  best solution fitness value:  0 

epoch # 3  best solution fitness value:  0 

epoch # 4  best solution fitness value:  0 

epoch # 5  best solution fitness value:  0 

epoch # 6  best solution fitness value:  0 

epoch # 7  best solution fitness value:  0 

epoch # 8  best solution fitness value:  0 

epoch # 9  best solution fitness value:  0 


Generating report for Benchmark GA solution:
--- Benchmark Genetic Algorithm Quality Report ---
Assigned exams: 800 / 800
Hard constraints:
  Same room-timeslot conflicts: 390
  Capacity violations: 185
  Student same-timeslot conflicts (overlap count): 2222

Soft metrics:
  Late exams: 0 / 800
  Average room utilization: 0.7558
  Min room utilization: 0.0037
  Max room utilization: 8.6
  Same-day overlap penalty: 6389

Timeslot load:
  1995-01-23 09:00 -> 30 exams
  1995-01-23 13:30 -> 21 exams
  1995-01-23 16: